<a href="https://colab.research.google.com/github/ahmadtariq2004/Evolvix-Ai-ML-Internship-ahmadtariq2004/blob/main/week-3/task-2/sentiment_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Transformers library install karein
!pip install transformers torch

# 2. Apna model load karein
from transformers import pipeline

print("Model load ho raha hai...")
pipe = pipeline("text-classification", model="ahmed792002/Finetuning_Longformer_IMDb_movie_reviews_Classification")

# 3. Model ko test karein
review = "This movie was absolutely fantastic and I loved the acting!"
result = pipe(review)

print("Result:", result)

Model load ho raha hai...


config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  595MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/273 [00:00<?, ?it/s]

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.56M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

[transformers] Initializing global attention on CLS token...
[transformers] Input ids are automatically padded to be a multiple of `config.attention_window`: 512


Result: [{'label': 'LABEL_1', 'score': 0.9990109205245972}]


In [6]:
# 1. Required packages install
!pip install --upgrade datasets huggingface_hub
!pip install scikit-learn

import os
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from transformers import pipeline

# 2. IMDb Dataset Load karna (Using standard HF parquet load which doesn't execute local Python code)
print("Dataset load ho raha hai...")
try:
    # Use the official dataset without local python script execution
    dataset = load_dataset("imdb")
except Exception as e:
    print("Standard load failed, trying standard parquet/raw fallback...")
    dataset = load_dataset("mteb/imdb")

# Shuffle the dataset to get a balanced set of labels (0 and 1) instead of only 0s
shuffled_train = dataset['train'].shuffle(seed=42)
shuffled_test = dataset['test'].shuffle(seed=42)

train_texts = shuffled_train['text'][:200]
train_labels = shuffled_train['label'][:200]
test_texts = shuffled_test['text'][:200]
test_labels = shuffled_test['label'][:200]

# 3. Classical ML (TF-IDF + Logistic Regression)
print("TF-IDF Model train ho raha hai...")
vectorizer = TfidfVectorizer(max_features=5000)
X_train = vectorizer.fit_transform(train_texts)
X_test = vectorizer.transform(test_texts)

clf = LogisticRegression()
clf.fit(X_train, train_labels)

tfidf_preds = clf.predict(X_test)
tfidf_acc = accuracy_score(test_labels, tfidf_preds)
tfidf_f1 = f1_score(test_labels, tfidf_preds, zero_division=0)

# 4. Hugging Face Model Testing (Initialize pipeline if not defined)
try:
    pipe
except NameError:
    print("Pipeline missing! Initializing Hugging Face pipeline...")
    pipe = pipeline("text-classification", model="ahmed792002/Finetuning_Longformer_IMDb_movie_reviews_Classification")

print("Hugging Face model predictions nikal raha hai (is mein 1 minute lag sakta hai)...")
hf_results = pipe(test_texts)

def map_hf_label(label_str):
    return 1 if '1' in label_str or 'pos' in label_str.lower() or 'positive' in label_str.lower() else 0

hf_preds = [map_hf_label(res['label']) for res in hf_results]
hf_acc = accuracy_score(test_labels, hf_preds)
hf_f1 = f1_score(test_labels, hf_preds, zero_division=0)

# 5. Final Result Comparison
print("\n" + "="*40)
print("FINAL ASSIGNMENT COMPARISON")
print("="*40)
print(f"Classical (TF-IDF) Model -> Accuracy: {tfidf_acc:.2f} | F1-Score: {tfidf_f1:.2f}")
print(f"Hugging Face Transformer -> Accuracy: {hf_acc:.2f} | F1-Score: {hf_f1:.2f}")
print("="*40)

  Using cached huggingface_hub-2.1.1-py3-none-any.whl.metadata (16 kB)
Dataset load ho raha hai...
Standard load failed, trying standard parquet/raw fallback...
TF-IDF Model train ho raha hai...
Pipeline missing! Initializing Hugging Face pipeline...


Loading weights:   0%|          | 0/273 [00:00<?, ?it/s]

[transformers] Initializing global attention on CLS token...
[transformers] Input ids are automatically padded to be a multiple of `config.attention_window`: 512


Hugging Face model predictions nikal raha hai (is mein 1 minute lag sakta hai)...

FINAL ASSIGNMENT COMPARISON
Classical (TF-IDF) Model -> Accuracy: 0.71 | F1-Score: 0.72
Hugging Face Transformer -> Accuracy: 0.94 | F1-Score: 0.94
